In [18]:
import torch
from util import TokenizerUtil

tokenizer = TokenizerUtil()

input_ids, attention_mask = tokenizer.encode('how are you', max_length = 4 )

input_ids, attention_mask, tokenizer.decode(input_ids), input_ids.shape

(tensor([   0, 9178,   32,    2]),
 tensor([1, 1, 1, 1]),
 '<s>how are</s>',
 torch.Size([4]))

In [19]:
from datasets import load_dataset
dataset = load_dataset('json', data_files='dataset/train.json', split='train')
#2,4,4切分,取第0部分
dataset = dataset.select(range(15000,45000))
len(dataset)

30000

In [20]:
def f(data):
    chosen = data['prompt'] + data['chosen'].swapcase()
    rejected = data['prompt'] + data['chosen']

    chosen_input_ids, chosen_attention_mask = tokenizer.encode(chosen)
    rejected_input_ids, rejected_attention_mask = tokenizer.encode(rejected)

    return {
        'chosen_input_ids': chosen_input_ids,
        'chosen_attention_mask': chosen_attention_mask,
        'rejected_input_ids': rejected_input_ids,
        'rejected_attention_mask': rejected_attention_mask
    }


dataset = dataset.map(f)

In [21]:
def f(data):
    # 1. 先准备好四个空列表
    chosen_input_ids = []
    chosen_attention_mask = []
    rejected_input_ids = []
    rejected_attention_mask = []
    
    # 2. 只遍历 1 次 data，同时给四个列表塞数据
    for i in data:
        chosen_input_ids.append(torch.tensor(i['chosen_input_ids']))
        chosen_attention_mask.append(torch.tensor(i['chosen_attention_mask']))
        rejected_input_ids.append(torch.tensor(i['rejected_input_ids']))
        rejected_attention_mask.append(torch.tensor(i['rejected_attention_mask']))

    input_ids = torch.stack(chosen_input_ids + rejected_input_ids, dim=0)
    attention_mask = torch.stack(chosen_attention_mask+rejected_attention_mask, dim=0)
    
    return {
        'input_ids': input_ids,
        'attention_mask':attention_mask
    }
    
loader = torch.utils.data.DataLoader(dataset, 
                                    collate_fn = f,
                                    batch_size = 4,
                                    shuffle = True,
                                    drop_last = True)

len(loader), next(iter(loader)), next(iter(loader))['input_ids'].shape

(7500,
 {'input_ids': tensor([[    0, 33837,    35,  ...,     1,     1,     1],
          [    0, 33837,    35,  ...,     1,     1,     1],
          [    0, 33837,    35,  ...,     1,     1,     1],
          ...,
          [    0, 33837,    35,  ...,     1,     1,     1],
          [    0, 33837,    35,  ...,     1,     1,     1],
          [    0, 33837,    35,  ...,     1,     1,     1]]),
  'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0],
          ...,
          [1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0]])},
 torch.Size([8, 512]))

In [22]:
from lora import count_params

class CriticModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        from transformers import AutoModel
        self.rwtransformer = AutoModel.from_pretrained("/root/autodl-tmp/StudyLLM/02. Deepseed/opt-350m",
                                                       torch_dtype=torch.bfloat16,
                                                       dropout=0.0)

        self.v_head = torch.nn.Linear(512, 1, bias=False).to(torch.bfloat16)   

    def forward(self, input_ids, attention_mask):
        value = self.rwtransformer(input_ids=input_ids, attention_mask = attention_mask).last_hidden_state
        value = self.v_head(value).squeeze(-1)

        loss_sum = 0.0
        value_chosen_sum = 0.0 
        value_rejected_sum = 0.0

        for input_ids_chosen, input_ids_rejected, value_chosen, value_rejected in zip(
            input_ids[:4], input_ids[4:], value[:4], value[4:]):

            start = (input_ids_chosen == input_ids_rejected).tolist().index(False)
            end_chosen = input_ids_chosen.tolist().index(tokenizer.eos_token_id)+1 # +1是为了应对左闭右开的特点
            end_rejected = input_ids_rejected.tolist().index(tokenizer.eos_token_id)+1 # +1是为了应对左闭右开的特点
            end = max(end_chosen, end_rejected)

            value_chosen = value_chosen[start:end]
            value_rejected = value_rejected[start:end]
            loss = value_chosen - value_rejected
            loss = -torch.nn.functional.logsigmoid(loss).mean()

            loss_sum += loss
            value_chosen_sum += value_chosen.mean().item()
            value_rejected_sum += value_rejected.mean().item()

        return loss_sum / 4, value_chosen_sum, value_rejected_sum   

In [23]:
model_critic = CriticModel()

count_params(model_critic)

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

{'count_require': 3.31196928, 'count_all': 3.31196928, 'ratio': 1.0}


In [24]:
from transformers import get_scheduler
from accelerate import Accelerator


def f():
    params_decay = []
    params = []
    for name, param in model_critic.named_parameters():
        if 'bias' in name or 'norm.weight' in name:
            params.append(param)
            continue
        params_decay.append(param)

    return [{
        'params': params_decay,
        'weight_decay': 0.1
    }, {
        'params': params,
        'weight_decay': 0.0
    }]


optimizer = torch.optim.Adam(f(), lr=5e-5, betas=(0.9, 0.95))

scheduler = get_scheduler(name='cosine',
                          optimizer=optimizer,
                          num_warmup_steps=0,
                          num_training_steps=500)

accelerator = Accelerator(gradient_accumulation_steps=16,
                          mixed_precision='bf16')

model_critic, loader, optimizer, scheduler = accelerator.prepare(
    model_critic, loader, optimizer, scheduler)

model_critic.train()

CriticModel(
  (rwtransformer): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_feature

In [25]:
for i, data in enumerate(loader):
    with accelerator.accumulate(model_critic):
        loss, value_chosen_sum, value_rejected_sum = model_critic(**data)
        accelerator.backward(loss)

        if accelerator.sync_gradients:
            accelerator.clip_grad_norm_(model_critic.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    if (i + 1) % 100 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(i, len(loader), loss.item(), lr, value_chosen_sum,
              value_rejected_sum)

    if i == 2000:
        break

torch.save(model_critic.to('cpu'), 'model/critic')

99 7500 0.05224609375 4.998223681601473e-05 18.796875 1.929931640625
199 7500 0.00177764892578125 4.992897250651535e-05 28.53125 -8.54296875
299 7500 0.0001277923583984375 4.984028276300021e-05 32.34375 -18.375
399 7500 2.6226043701171875e-05 4.9692208514878444e-05 35.1875 -17.8359375
499 7500 5.91278076171875e-05 4.952726293608335e-05 30.8125 -19.25
599 7500 0.0009765625 4.9327462774553166e-05 28.84375 -8.453125
699 7500 0.0091552734375 4.909309195725025e-05 31.09375 -3.0859375
799 7500 5.3882598876953125e-05 4.877641290737884e-05 27.90625 -17.34375
899 7500 8.440017700195312e-05 4.846834644134686e-05 29.84375 -18.25
999 7500 2.586841583251953e-05 4.812693017086145e-05 28.78125 -21.90625
1099 7500 2.7894973754882812e-05 4.775264926712489e-05 27.96875 -23.8125
1199 7500 0.0003833770751953125 4.72751631047092e-05 29.25 -18.6875
1299 7500 0.002960205078125 4.683156137024801e-05 26.5 -17.451171875
1399 7500 9.715557098388672e-06 4.635693579248238e-05 27.5 -26.71875
1499 7500 9.77516174316